In [1]:
import pandas as pd
import numpy as np

In [2]:
# Data Initialization
df = pd.read_csv('Data/data_capstone_ecommerce.csv', sep='|')
df.columns = df.columns.str.strip()

In [3]:
# Handling Duplicates
df.drop_duplicates(subset=['Customer_Name', 'Product_Category', 'Order_Date'], inplace=True)

# Keseluruhan menghapus whitespace/spasi yang ada pada setiap kolom yang tercantum
df['Customer_Name'] = df['Customer_Name'].str.strip()
df['Product_Category'] = df['Product_Category'].str.strip()
df['Price_per_Unit'] = df['Price_per_Unit'].str.strip()
df['Quantity'] = df['Quantity'].str.strip()
df['Order_Date'] = df['Order_Date'].str.strip()
df['Discount_Code'] = df['Discount_Code'].str.strip()
df['Shipping_Cost'] = df['Shipping_Cost'].str.strip()

df.head()

,Order_ID,Customer_Name,Product_Category,Price_per_Unit,Quantity,Order_Date,Discount_Code,Shipping_Cost
0,ORD-001,Budi Santoso,Electronics,Rp 1.500.000,2 pcs,2024-02-01,PROMO20,Rp 50.000
1,ord-002,siti aminah,fashion,"IDR 250,000",1,02/02/2024,NONE,Gratis
2,ORD-003,Andi T.,ELEC,3.5 Juta,-1,Feb 03 2024,NaN,100K
4,ORD-004,Joko,Groceries,Rp 50.000,5,2024-02-04,DISKON50,Rp 15.000
5,ORD-005,Rina,FASHION,150000,NaN,05-02-2024,promo20,20k


In [4]:
# Text Standardization
# Mengubah tulisan pada kolom 'Order_ID' dengan kapital keseluruhan
df['Order_ID'] = df['Order_ID'].str.upper() 
# Mengubah tulisan pada kolom 'Customer_Name' dengan kapital per-kata
df['Customer_Name'] = df['Customer_Name'].str.title() 
# Ubah kata 'Elec' dengan 'Electronics' -> ubah tulisan pada kolom 'Product_Category' dengan kapital per-kata
df['Product_Category'] = df['Product_Category'].str.upper().replace('ELEC', 'ELECTRONICS').str.title()
# Ubah ke huruf kapital keseluruhan -> ganti kata 'NONE' dengan np.nan (NaN)
df['Discount_Code'] = df['Discount_Code'].str.upper().replace(['NONE', 'NAN'], np.nan) 

df.head()

,Order_ID,Customer_Name,Product_Category,Price_per_Unit,Quantity,Order_Date,Discount_Code,Shipping_Cost
0,ORD-001,Budi Santoso,Electronics,Rp 1.500.000,2 pcs,2024-02-01,PROMO20,Rp 50.000
1,ORD-002,Siti Aminah,Fashion,"IDR 250,000",1,02/02/2024,NaN,Gratis
2,ORD-003,Andi T.,Electronics,3.5 Juta,-1,Feb 03 2024,NaN,100K
4,ORD-004,Joko,Groceries,Rp 50.000,5,2024-02-04,DISKON50,Rp 15.000
5,ORD-005,Rina,Fashion,150000,NaN,05-02-2024,PROMO20,20k


In [5]:
# Complex Currency Formatting
def currencyFormatting(teks) :
    # isinstance untuk mengecek apakah data berupa teks
    if isinstance(teks, str) :
        teks = teks.upper().strip() # Mengubah menjadi huruf kapital semua agar seragam, lalu hapus spasi
        
        # Jika terdapat kata JUTA, maka akan mengembalikan bilangan pecahan angka_bersih dengan mengalikan 1 Juta
        if 'JUTA' in teks :
            # Menghapus kata juta didalam teks (jika ada)
            angka_bersih = teks.replace('JUTA', '').replace('RP', '').replace('IDR', '').strip()
            return float(angka_bersih) * 1000000 # Return dengan mengkonversi ke tipe data float, lalu kali dengan 1 juta
        else :
            """
            Ganti kata 'RP' dengan string kosong -> 
            ganti kata 'IDR' dengan string kosong -> 
            ganti kata '.' dengan string kosong -> 
            ganti kata ',' dengan string kosong ->
            hapus whitespace/spasi
            """
            angka_bersih = teks.replace('RP', '').replace('IDR', '').replace('.', '').replace(',', '').strip()
            return float(angka_bersih)
    
    # Jika data sudah berupa angka (float/int) dari awal, kembalikan apa adanya
    return float(teks) if pd.notnull(teks) else np.nan

def shippingFormatting(teks) :
    # isinstance untuk mengecek apakah data berupa teks
    if isinstance(teks, str) :
        teks = teks.upper().strip() # Mengubah menjadi huruf kapital semua agar seragam, lalu hapus spasi

        # Jika terdapat kata 'GRATIS' didalam teks, maka akan mengembalika 0 dengan bilangan pecahan
        if 'GRATIS' in teks :
            return 0.0
        # Jika terdapat huruf 'K' didalam teks, maka akan mengembalikan bilangan pecahan 'angka_bersih' dengan mengalikan seribu
        elif 'K' in teks :
            angka_bersih = teks.replace('K', '').strip()
            return float(angka_bersih) * 1000
        else :
            # Ganti kata 'Rp' dengan string kosong -> ganti kata '.' dengan string kosong -> ganti kata ',' dengan string kosong -> hapus whitespace/spasi
            angka_bersih = teks.replace('RP', '').replace('.', '').replace(',', '').strip()
            return float(angka_bersih)
    
    # Jika data sudah berupa angka (float/int) dari awal, kembalikan apa adanya
    return float(teks) if pd.notnull(teks) else np.nan

df['Price_per_Unit'] = df['Price_per_Unit'].apply(currencyFormatting) # Menerapkan fungsi kustom ke kolom 'Price_per_Unit' pada Dataframe
df['Shipping_Cost'] = df['Shipping_Cost'].apply(shippingFormatting) # Menerapkan fungsi kustom ke kolom 'Shipping_Cost' pada Dataframe

df.head()

,Order_ID,Customer_Name,Product_Category,Price_per_Unit,Quantity,Order_Date,Discount_Code,Shipping_Cost
0,ORD-001,Budi Santoso,Electronics,1500000.0,2 pcs,2024-02-01,PROMO20,50000.0
1,ORD-002,Siti Aminah,Fashion,250000.0,1,02/02/2024,NaN,0.0
2,ORD-003,Andi T.,Electronics,3500000.0,-1,Feb 03 2024,NaN,100000.0
4,ORD-004,Joko,Groceries,50000.0,5,2024-02-04,DISKON50,15000.0
5,ORD-005,Rina,Fashion,150000.0,NaN,05-02-2024,PROMO20,20000.0


In [6]:
# Logical Constraints & Imputation
# konversi ke tipe data str -> ganti kata 'pcs' dengan string kosong -> konversi lagi ke tipe data float
df['Quantity'] = df['Quantity'].astype(str).str.replace('pcs', '').astype(float) 
df['Quantity'] = df['Quantity'].abs() # Mengubah kolom 'Quantity' menjadi absolut

median_quantity = df['Quantity'].median() # Mencari nilai median pada kolom 'Quantity'
df.fillna({'Quantity': median_quantity}, inplace=True) # Menyisipkan/memasukkan nilai pada kolom 'Quantity' dengan variabel median_quantity

df['Quantity'] = df['Quantity'].astype(int) # Mengkonversi kolom 'Quantity' ke tipe data int
df.head()

,Order_ID,Customer_Name,Product_Category,Price_per_Unit,Quantity,Order_Date,Discount_Code,Shipping_Cost
0,ORD-001,Budi Santoso,Electronics,1500000.0,2,2024-02-01,PROMO20,50000.0
1,ORD-002,Siti Aminah,Fashion,250000.0,1,02/02/2024,NaN,0.0
2,ORD-003,Andi T.,Electronics,3500000.0,1,Feb 03 2024,NaN,100000.0
4,ORD-004,Joko,Groceries,50000.0,5,2024-02-04,DISKON50,15000.0
5,ORD-005,Rina,Fashion,150000.0,2,05-02-2024,PROMO20,20000.0


In [7]:
# Date Parsing
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='mixed', dayfirst=True) # Mengkonversi kedalam satuan tanggal waktu
df.head()

,Order_ID,Customer_Name,Product_Category,Price_per_Unit,Quantity,Order_Date,Discount_Code,Shipping_Cost
0,ORD-001,Budi Santoso,Electronics,1500000.0,2,2024-02-01,PROMO20,50000.0
1,ORD-002,Siti Aminah,Fashion,250000.0,1,2024-02-02,NaN,0.0
2,ORD-003,Andi T.,Electronics,3500000.0,1,2024-02-03,NaN,100000.0
4,ORD-004,Joko,Groceries,50000.0,5,2024-02-04,DISKON50,15000.0
5,ORD-005,Rina,Fashion,150000.0,2,2024-02-05,PROMO20,20000.0


In [8]:
# Feature Engineering
# Mengkalikan kolom 'Price_per_Unit' dan 'Quantity' dengan membuat kolom baru yaitu 'Total_Sales'
df['Total_Sales'] = df['Price_per_Unit'] * df['Quantity'] 
# Menjumlahkan kolom 'Total_Sales' dan 'Shipping_Cost' dengan mmembuat kolom baru yaitu 'Final_Amount'
df['Final_Amount'] = df['Total_Sales'] + df['Shipping_Cost']

df.head()

,Order_ID,Customer_Name,Product_Category,Price_per_Unit,Quantity,Order_Date,Discount_Code,Shipping_Cost,Total_Sales,Final_Amount
0,ORD-001,Budi Santoso,Electronics,1500000.0,2,2024-02-01,PROMO20,50000.0,3000000.0,3050000.0
1,ORD-002,Siti Aminah,Fashion,250000.0,1,2024-02-02,NaN,0.0,250000.0,250000.0
2,ORD-003,Andi T.,Electronics,3500000.0,1,2024-02-03,NaN,100000.0,3500000.0,3600000.0
4,ORD-004,Joko,Groceries,50000.0,5,2024-02-04,DISKON50,15000.0,250000.0,265000.0
5,ORD-005,Rina,Fashion,150000.0,2,2024-02-05,PROMO20,20000.0,300000.0,320000.0


In [9]:
# Final & Export
df = df.reset_index(drop=True)
df.to_csv('cleaned_ecommerce.csv', index=False) # Menyimpan Dataframe yang sudah bersih kedalam file csv

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Order_ID          18 non-null     object        
 1   Customer_Name     18 non-null     object        
 2   Product_Category  18 non-null     object        
 3   Price_per_Unit    18 non-null     float64       
 4   Quantity          18 non-null     int64         
 5   Order_Date        18 non-null     datetime64[ns]
 6   Discount_Code     9 non-null      object        
 7   Shipping_Cost     18 non-null     float64       
 8   Total_Sales       18 non-null     float64       
 9   Final_Amount      18 non-null     float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(4)
memory usage: 1.5+ KB
